# Incremental SFT — Continue from Existing LoRA

Loads the LoRA from the first training run and continues training with new + old data.
Uses lower learning rate (5e-5) to refine without catastrophic forgetting.

## 1. Load Existing LoRA

In [2]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048

# Load the LoRA adapter from the first training run
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="./outputs/s1_sft/final_lora",
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,
)

free, total = torch.cuda.mem_get_info(0)
print(f"VRAM: {(total-free)/1e9:.1f} GB used / {total/1e9:.1f} GB total")
print(f"Loaded LoRA from ./outputs/s1_sft/final_lora")

[unsloth.import_fixes|WARNING]Unsloth: torch==2.12.0.dev20260323+cu128 requires torchvision>=0.27.0, but found torchvision==0.26.0.dev20260323+cu128. Try updating torchvision via `pip install --upgrade "torchvision>=0.27.0"`. Please refer to https://pytorch.org/get-started/previous-versions/ for more information.
Detected a pre-release build. Continuing with a warning. Set UNSLOTH_SKIP_TORCHVISION_CHECK=1 to silence this.


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


c:\Users\sandy\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0324 12:12:24.667000 14984 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
c:\Users\sandy\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchao\float8\float8_training_tensor.py:122: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
c:\Users\sandy\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchao\float8\float8_training_tensor.py:195: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  

Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.10: Fast Qwen3_5 patching. Transformers: 5.3.0.
   \\   /|    NVIDIA GeForce RTX 5070 Ti. Num GPUs = 1. Max memory: 15.92 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.12.0.dev20260323+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights:   0%|          | 2/760 [00:01<11:47,  1.07it/s]c:\Users\sandy\AppData\Local\Programs\Python\Python312\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 760/760 [00:10<00:00, 72.56it/s] 


VRAM: 12.3 GB used / 17.1 GB total
Loaded LoRA from ./outputs/s1_sft/final_lora


## 2. Load & Merge Datasets

In [3]:
from datasets import load_dataset, concatenate_datasets
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
TD = PROJECT_ROOT / "training_data"

# Load original training data
original = load_dataset("json", data_files=str(TD / "s1_extraction.jsonl"), split="train")
print(f"Original: {len(original)} examples")

# Load supplementary training data
supp_path = TD / "s1_suplementary_training.jsonl"
if supp_path.exists():
    supplementary = load_dataset("json", data_files=str(supp_path), split="train")
    print(f"Supplementary: {len(supplementary)} examples")
else:
    supplementary = None
    print("No supplementary data found")

# Load stress test as additional training data (these are the edge cases we want to nail)
stress_path = TD / "s1_stress_test.jsonl"
if stress_path.exists():
    stress = load_dataset("json", data_files=str(stress_path), split="train")
    print(f"Stress test (as training): {len(stress)} examples")
else:
    stress = None
    print("No stress test data found")

# Merge all datasets
datasets_to_merge = [original]
if supplementary is not None:
    datasets_to_merge.append(supplementary)
if stress is not None:
    datasets_to_merge.append(stress)

train_dataset = concatenate_datasets(datasets_to_merge)
print(f"\nTotal training: {len(train_dataset)} examples")

# Validation
val_dataset = load_dataset("json", data_files=str(TD / "s1_validation.jsonl"), split="train")
print(f"Validation: {len(val_dataset)} examples")

Original: 450 examples
Supplementary: 112 examples
Stress test (as training): 20 examples

Total training: 582 examples
Validation: 50 examples


## 3. Format for Training

In [3]:
def format_example(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
        enable_thinking=False,
    )
    return {"text": text}

train_formatted = train_dataset.map(format_example)
val_formatted = val_dataset.map(format_example)

print(f"Formatted {len(train_formatted)} train, {len(val_formatted)} val")

Map: 100%|██████████| 50/50 [00:00<00:00, 1627.36 examples/s]

Formatted 582 train, 50 val


## 4. Train (Incremental)

In [4]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_formatted,
    eval_dataset=val_formatted,
    args=SFTConfig(
        output_dir="./outputs/s1_sft_v2",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=2,           # fewer epochs for refinement
        learning_rate=5e-5,           # lower LR to avoid forgetting
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=50,
        save_strategy="steps",
        save_steps=50,
        save_total_limit=3,
        optim="adamw_8bit",
        seed=42,
        max_seq_length=MAX_SEQ_LENGTH,
        dataset_text_field="text",
        dataset_num_proc=None,
        dataloader_num_workers=0,
        report_to="none",
    ),
)

print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"Training {len(train_formatted)} examples x 2 epochs")

Unsloth: Tokenizing ["text"]: 100%|██████████| 50/50 [00:00<00:00, 743.90 examples/s]

Trainable params: 29,097,984
Training 582 examples x 2 epochs


In [5]:
stats = trainer.train()
print(f"\nTraining complete.")
print(f"  Total steps: {stats.global_step}")
print(f"  Train loss:  {stats.training_loss:.4f}")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 582 | Num Epochs = 2 | Total steps = 146
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,097,984 of 9,438,911,728 (0.31% trained)


Unsloth: Will smartly offload gradients to save VRAM!


c:\Users\sandy\AppData\Local\Programs\Python\Python312\Lib\site-packages\bitsandbytes\_ops.py:239: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
c:\Users\sandy\AppData\Local\Programs\Python\Python312\Lib\site-packages\bitsandbytes\_ops.py:186: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
c:\Users\sandy\AppData\Local\Programs\Python\Python312\Lib\site-packages\bitsandbytes\_ops.py:239: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
c:\Users\sandy\AppData\Local\Programs\Python\Python312\Lib\site-packages\bitsandbytes\_ops.py:186: FutureWarning: _check_is_size will be removed in a future PyTorch release along with 

Step,Training Loss,Validation Loss
50,0.107801,0.113669
100,0.082281,0.115528


Unsloth: Not an error, but Qwen3_5ForConditionalGeneration does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient



Training complete.
  Total steps: 146
  Train loss:  0.1009


## 5. Quick Test

In [ ]:
import json

FastLanguageModel.for_inference(model)

test_messages = [
    {"role": "system", "content": "You are a knowledge extractor for a personal knowledge graph. Analyze the conversation and return a single JSON object with topic classification, entities, relations, and facts. Output valid JSON only, no markdown, no explanation."},
    {"role": "user", "content": "EXISTING NODES:\n[]\n\nTOPIC HINT: unresolved \u2014 classify the topic yourself\nCURRENT TOPIC: null\n\nPREVIOUS ASSISTANT: null\nUSER: Estoy trabajando en un proyecto llamado Atlas con React y FastAPI. Usamos PostgreSQL."}
]

inner_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)
text = inner_tokenizer.apply_chat_template(
    test_messages, tokenize=False, add_generation_prompt=True, enable_thinking=False,
)
inputs = inner_tokenizer(text, return_tensors="pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens=512, temperature=0.1, do_sample=True)
response = inner_tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

print("--- Model Output ---")
try:
    parsed = json.loads(response)
    print(json.dumps(parsed, indent=2, ensure_ascii=False))
except json.JSONDecodeError as e:
    print(response)
    print(f"\nJSON parse error: {e}")

c:\Users\sandy\AppData\Local\Programs\Python\Python312\Lib\site-packages\bitsandbytes\_ops.py:239: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
c:\Users\sandy\AppData\Local\Programs\Python\Python312\Lib\site-packages\bitsandbytes\_ops.py:186: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
c:\Users\sandy\AppData\Local\Programs\Python\Python312\Lib\site-packages\bitsandbytes\_ops.py:239: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
c:\Users\sandy\AppData\Local\Programs\Python\Python312\Lib\site-packages\bitsandbytes\_ops.py:186: FutureWarning: _check_is_size will be removed in a future PyTorch release along with 

--- Model Output ---
{
  "topic": {
    "action": "changed",
    "label": "Atlas development"
  },
  "entities": [
    {
      "id": "atlas",
      "label": "Atlas",
      "type": "project",
      "layer": "PERSONAL",
      "attributes": {},
      "facts": [],
      "existing_id": null
    },
    {
      "id": "react",
      "label": "React",
      "type": "technology",
      "layer": "UNIVERSAL",
      "attributes": {},
      "facts": [],
      "existing_id": null
    },
    {
      "id": "fastapi",
      "label": "FastAPI",
      "type": "technology",
      "layer": "UNIVERSAL",
      "attributes": {},
      "facts": [],
      "existing_id": null
    },
    {
      "id": "postgresql",
      "label": "PostgreSQL",
      "type": "technology",
      "layer": "UNIVERSAL",
      "attributes": {},
      "facts": [],
      "existing_id": null
    },
    {
      "id": "luca_rossi",
      "label": "Luca Rossi",
      "type": "person",
      "layer": "PERSONAL",
      "attributes": {},
      "

## 6. Stress Test

In [8]:
import json
from pathlib import Path

stress_path = Path.cwd().parent / "training_data" / "s1_stress_test.jsonl"
if not stress_path.exists():
    stress_path = Path("training_data/s1_stress_test.jsonl")

with open(stress_path, encoding="utf-8") as f:
    stress_examples = [json.loads(line) for line in f]

print(f"Running {len(stress_examples)} stress test cases...\n")

inner_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)
FastLanguageModel.for_inference(model)

results = {"pass": 0, "json_fail": 0, "wrong": 0}
failures = []

for i, example in enumerate(stress_examples):
    messages = example["messages"][:2]
    expected = json.loads(example["messages"][2]["content"])

    text = inner_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )
    inputs = inner_tokenizer(text, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=512, temperature=0.1, do_sample=True)
    response = inner_tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

    try:
        parsed = json.loads(response)
    except json.JSONDecodeError:
        results["json_fail"] += 1
        failures.append((i + 1, "JSON_FAIL", response[:100]))
        continue

    ok = True
    issues = []

    expected_action = expected["topic"]["action"] if isinstance(expected["topic"], dict) else expected["topic"].get("action")
    got_action = parsed.get("topic", {}).get("action")
    if got_action != expected_action:
        issues.append(f"topic: expected={expected_action}, got={got_action}")
        ok = False

    expected_has_entities = len(expected.get("entities", [])) > 0
    got_has_entities = len(parsed.get("entities", [])) > 0
    if expected_has_entities != got_has_entities:
        issues.append(f"entities: expected {'non-empty' if expected_has_entities else 'empty'}, got {'non-empty' if got_has_entities else 'empty'}")
        ok = False

    expected_has_facts = len(expected.get("facts", [])) > 0
    got_has_facts = len(parsed.get("facts", [])) > 0
    if expected_has_facts != got_has_facts:
        issues.append(f"facts: expected {'non-empty' if expected_has_facts else 'empty'}, got {'non-empty' if got_has_facts else 'empty'}")
        ok = False

    if ok:
        results["pass"] += 1
    else:
        results["wrong"] += 1
        user_msg = messages[1]["content"].split("USER: ")[-1][:60]
        failures.append((i + 1, "; ".join(issues), user_msg))

total = len(stress_examples)
print(f"Results: {results['pass']}/{total} pass, {results['json_fail']} JSON failures, {results['wrong']} wrong")
print(f"JSON parse rate: {(total - results['json_fail'])/total:.0%}")
print(f"Accuracy: {results['pass']/total:.0%}")

if failures:
    print(f"\n--- Failures ---")
    for num, issue, ctx in failures:
        print(f"  #{num}: {issue}")
        print(f"       {ctx}")

Running 20 stress test cases...

Results: 17/20 pass, 0 JSON failures, 3 wrong
JSON parse rate: 100%
Accuracy: 85%

--- Failures ---
  #14: entities: expected empty, got non-empty; facts: expected non-empty, got empty
       Queremos agregar caching a la app principal. Vamos a integra
  #17: facts: expected non-empty, got empty
       Estamos deploying en AWS con Docker. También estamos pensand
  #20: facts: expected empty, got non-empty
       Usamos TypeScript con React. También tenemos ESLint y Pretti


## 7. Save Model

In [9]:
model.save_pretrained("./outputs/s1_sft_v2/final_lora")
tokenizer.save_pretrained("./outputs/s1_sft_v2/final_lora")
print("LoRA v2 saved to ./outputs/s1_sft_v2/final_lora")

LoRA v2 saved to ./outputs/s1_sft_v2/final_lora


In [5]:
# Optional: Export to GGUF (requires CMake in PATH — restart kernel if needed)
model.save_pretrained_gguf(
    "./outputs/s1_sft_v2/gguf",
    tokenizer,
    quantization_method="q4_k_m",
)
print("GGUF v2 exported")

Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: C:\Users\sandy\.cache\huggingface\hub


Fetching 1 files: 100%|██████████| 1/1 [00:00<00:00, 84.92it/s]


Checking cache directory for required files...


Unsloth: Copying 4 files from cache to `./outputs/s1_sft_v2/gguf`: 100%|██████████| 4/4 [00:16<00:00,  4.15s/it]


Successfully copied all 4 files from cache to `./outputs/s1_sft_v2/gguf`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [00:39<00:00,  9.88s/it]


Unsloth: Merge process complete. Saved to `d:\Development\acervo-graph-model\02_training\outputs\s1_sft_v2\gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: llama.cpp folder exists but binaries not found - will build
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


[unsloth_zoo.llama_cpp|WARNING]Unsloth: Qwen2MoE num_experts patch target not found.


Unsloth: [1] Converting model into bf16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['./outputs/s1_sft_v2/gguf_gguf\\Qwen3.5-9B.BF16.gguf', './outputs/s1_sft_v2/gguf_gguf\\Qwen3.5-9B.BF16-mmproj.gguf']
Unsloth: [2] Converting GGUF bf16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['./outputs/s1_sft_v2/gguf_gguf\\Qwen3.5-9B.Q4_K_M.gguf', './outputs/s1_sft_v2/gguf_gguf\\Qwen3.5-9B.BF16-mmproj.gguf']
Unsloth: No Ollama template mapping found for model 'unsloth/Qwen3.5-9B'. Skipping Ollama Modelfile


Unsloth: example usage for Multimodal LLMs: C:\Users\sandy\.unsloth\llama.cpp\build\bin\Release\llama-mtmd-cli.exe -m ./outputs/s1_sft_v2/gguf_gguf\Qwen3.5-9B.Q4_K_M.gguf --mmproj ./outputs/s1_sft_v2/gguf_gguf\Qwen3.5-9B.BF16-mmproj.gguf
Unsloth: load image inside llama.cpp runner: /image test_image.jpg
Unsloth: Prompt model to describe the image
GGU